# Loading dataset and preprocessing

In [11]:
import pandas as pd
df = pd.read_json("news_dataset.json")
df.head()

,text,category
0,Watching Schrödinger's Cat Die University of C...,SCIENCE
1,WATCH: Freaky Vortex Opens Up In Flooded Lake,SCIENCE
2,Entrepreneurs Today Don't Need a Big Budget to...,BUSINESS
3,These Roads Could Recharge Your Electric Car A...,BUSINESS
4,Civilian 'Guard' Fires Gun While 'Protecting' ...,CRIME


In [12]:
# The classes are not balanced
df['category'].value_counts()

,count
category,
BUSINESS,4254
SPORTS,4167
CRIME,2893
SCIENCE,1381


In [13]:
# Handling class imbalance using undersampling
min_samples = 1381
df_business = df[df['category']=="BUSINESS"].sample(min_samples, random_state=42)
df_sports = df[df['category']=="SPORTS"].sample(min_samples, random_state=42)
df_crime = df[df['category']=="CRIME"].sample(min_samples, random_state=42)
df_science = df[df['category']=="SCIENCE"].sample(min_samples, random_state=42)

df = pd.concat([df_business, df_sports, df_crime, df_science], axis=0)
df["category"].value_counts()

,count
category,
BUSINESS,1381
SPORTS,1381
CRIME,1381
SCIENCE,1381


In [14]:
# Convert text category to number
df["category_num"] = df["category"].map(
    {"BUSINESS": 0,
     "SPORTS": 1,
     "CRIME": 2,
     "SCIENCE": 3
     })

df.sample(5)

,text,category,category_num
6865,Scientists To Study Mysterious Interstellar As...,SCIENCE,3
3348,Abdul Malik Abdul Kareem Guilty Of Conspiring ...,CRIME,2
2853,Why you Get so Pumped up Watching Sports Sport...,SCIENCE,3
2309,How Marketers Should Appeal to Women Marketers...,BUSINESS,0
5304,Major Companies Back Obama’s Climate Regulatio...,BUSINESS,0


In [15]:
# Preprocess function
import spacy
nlp = spacy.load("en_core_web_sm", disable=["parser", "tagger", "ner"])

def preprocess(texts):
    results = []
    for doc in nlp.pipe(texts, batch_size=1000):
        tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
        results.append(" ".join(tokens))
    return results

In [16]:
df["preprocessed_txt"] = preprocess(df["text"])

/usr/local/lib/python3.12/dist-packages/spacy/pipeline/lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


In [17]:
df.head()

,text,category,category_num,preprocessed_txt
594,How to Develop the Next Generation of Innovato...,BUSINESS,0,develop generation innovators stop treating wa...
3093,"Madoff Victims' Payout Nears $7.2 Billion, Tru...",BUSINESS,0,madoff victims payout nears $ 7.2 billion trus...
7447,Bay Area Floats 'Sanctuary In Transit Policy' ...,BUSINESS,0,bay area floats sanctuary transit policy prote...
10388,Microsoft Agrees To Acquire LinkedIn For $26.2...,BUSINESS,0,microsoft agrees acquire linkedin $ 26.2 billi...
1782,"Inside A Legal, Multibillion Dollar Weed Market",BUSINESS,0,inside legal multibillion dollar weed market


In [18]:
# Train test split
from sklearn.model_selection import train_test_split
X = df["text"]
y = df["category_num"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=df_balanced["category_num"])

# Applying BoW and naive bayes

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

clf = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('nb', MultinomialNB())
])

clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.869683257918552

In [20]:
from sklearn.metrics import classification_report
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.78      0.92      0.84       276
           1       0.92      0.85      0.88       276
           2       0.91      0.89      0.90       277
           3       0.89      0.82      0.85       276

    accuracy                           0.87      1105
   macro avg       0.88      0.87      0.87      1105
weighted avg       0.88      0.87      0.87      1105



# Applying n-grams and naive bayes

In [21]:
clf = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1,2))),
    ('nb', MultinomialNB())
])

clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.8588235294117647

In [22]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.73      0.95      0.82       276
           1       0.92      0.83      0.87       276
           2       0.91      0.88      0.89       277
           3       0.93      0.79      0.85       276

    accuracy                           0.86      1105
   macro avg       0.87      0.86      0.86      1105
weighted avg       0.87      0.86      0.86      1105



# Applying TF-IDF and Naive Bayes

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

clf = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('nb', MultinomialNB())
])

clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.8660633484162896

In [24]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.78      0.93      0.85       276
           1       0.91      0.84      0.87       276
           2       0.91      0.89      0.90       277
           3       0.89      0.81      0.85       276

    accuracy                           0.87      1105
   macro avg       0.87      0.87      0.87      1105
weighted avg       0.87      0.87      0.87      1105



### Making predictions

In [25]:
X_test[:3]

,text
12446,Shocking Video Of Officer Punching Woman Ignit...
3868,YOLO ATTACK: Birthday Stabbing Turns Party Int...
3301,Great News For Obamacare


In [26]:
y_test[:3]

,category_num
12446,2
3868,2
3301,0


In [27]:
y_pred[:3]

array([2, 2, 0])

# Using spacy word vectors

In [29]:
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 1.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [30]:
import spacy
nlp = spacy.load("en_core_web_lg", disable=["parser", "ner", "tagger"])

In [31]:
vectors = []
for doc in nlp.pipe(df["preprocessed_txt"], batch_size=1000):
    vectors.append(doc.vector)

df["vector"] = vectors

/usr/local/lib/python3.12/dist-packages/spacy/pipeline/lemmatizer.py:188: UserWarning: [W108] The rule-based lemmatizer did not find POS annotation for one or more tokens. Check that your pipeline includes components that assign token.pos, typically 'tagger'+'attribute_ruler' or 'morphologizer'.
  warnings.warn(Warnings.W108)


In [32]:
df.head(3)

,text,category,category_num,preprocessed_txt,vector
594,How to Develop the Next Generation of Innovato...,BUSINESS,0,develop generation innovators stop treating wa...,"[-0.17358652, 0.11157323, -0.067405865, 0.0007..."
3093,"Madoff Victims' Payout Nears $7.2 Billion, Tru...",BUSINESS,0,madoff victims payout nears $ 7.2 billion trus...,"[-0.36055234, 0.24859288, 0.110320434, -0.0481..."
7447,Bay Area Floats 'Sanctuary In Transit Policy' ...,BUSINESS,0,bay area floats sanctuary transit policy prote...,"[0.057732146, -0.15535739, -0.03802716, -0.081..."


In [33]:
# Train test split
from sklearn.model_selection import train_test_split
X = df["vector"]
y = df["category_num"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [34]:
import numpy as np
X_train = np.stack(X_train)
X_test = np.stack(X_test)

In [37]:
# Applying Decision tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline

clf = Pipeline([
    ('Scaler', MinMaxScaler()),
    ('dt', DecisionTreeClassifier())
])

clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.6633484162895927

In [38]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.67      0.61      0.64       298
           1       0.63      0.59      0.61       270
           2       0.67      0.73      0.70       250
           3       0.68      0.72      0.70       287

    accuracy                           0.66      1105
   macro avg       0.66      0.66      0.66      1105
weighted avg       0.66      0.66      0.66      1105



In [35]:
# Applying Naive bayes
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline

clf = Pipeline([
    ('Scaler', MinMaxScaler()),
    ('nb', MultinomialNB())
])

clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.8416289592760181

In [36]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.80      0.83      0.81       298
           1       0.87      0.82      0.84       270
           2       0.83      0.89      0.86       250
           3       0.87      0.83      0.85       287

    accuracy                           0.84      1105
   macro avg       0.84      0.84      0.84      1105
weighted avg       0.84      0.84      0.84      1105



In [39]:
# Applying KNN
from sklearn.neighbors import KNeighborsClassifier
clf = Pipeline([
    ('Scaler', MinMaxScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5, metric='euclidean'))
])

clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.8343891402714932

In [40]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.75      0.89      0.81       298
           1       0.94      0.78      0.85       270
           2       0.80      0.90      0.85       250
           3       0.91      0.77      0.83       287

    accuracy                           0.83      1105
   macro avg       0.85      0.84      0.84      1105
weighted avg       0.85      0.83      0.84      1105



In [41]:
# Applying Random forest
from sklearn.ensemble import RandomForestClassifier
clf = Pipeline([
    ('Scaler', MinMaxScaler()),
    ('knn', RandomForestClassifier())
])

clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.860633484162896

In [42]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.84      0.81      0.83       298
           1       0.90      0.84      0.87       270
           2       0.85      0.90      0.88       250
           3       0.85      0.90      0.87       287

    accuracy                           0.86      1105
   macro avg       0.86      0.86      0.86      1105
weighted avg       0.86      0.86      0.86      1105



In [43]:
# Applying GradientBosstingClassifier
from sklearn.ensemble import GradientBoostingClassifier
clf = Pipeline([
    ('Scaler', MinMaxScaler()),
    ('gb', GradientBoostingClassifier())
])

clf.fit(X_train, y_train)
clf.score(X_test, y_test)

0.8723981900452489

In [44]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.85      0.85       298
           1       0.92      0.84      0.88       270
           2       0.87      0.90      0.88       250
           3       0.85      0.90      0.88       287

    accuracy                           0.87      1105
   macro avg       0.87      0.87      0.87      1105
weighted avg       0.87      0.87      0.87      1105

